# SentinelPay: Exploratory Data Analysis (EDA)
## Notebook 01 — Understanding Credit Card Fraud Patterns and Class Imbalance

**Author:** SentinelPay Research Team  
**Dataset:** `sentinelpay_benchmark_transactions.csv` (60,000 synthetic transactions, ~1.2% fraud rate)  
**Objective:** Explore feature distributions, quantify class imbalance severity, identify multivariate fraud signatures, and justify the need for specialized imbalance-handling techniques.

---

### 1. Introduction

Credit card fraud detection is a **binary classification problem** under extreme class imbalance. In production systems, fraudulent transactions typically constitute fewer than 2% of all activity (Pozzolo et al., 2015). This creates a deceptive accuracy trap where a naive model predicting *all transactions as legitimate* would achieve >98% accuracy while catching zero fraud.

This notebook establishes the statistical foundation for SentinelPay by:
1. Loading and inspecting the benchmark dataset.
2. Quantifying the exact class imbalance ratio.
3. Comparing feature distributions between legitimate and fraudulent transactions.
4. Computing correlation matrices and identifying non-linear relationships.
5. Formulating preprocessing and modeling strategies based on findings.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load benchmark dataset
df = pd.read_csv('../data/raw/sentinelpay_benchmark_transactions.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
df.info()

In [ ]:
# First 5 rows
df.head()

### 2. Class Imbalance Inspection

The target variable `is_fraud` encodes the ground truth label:
- **0** = Legitimate transaction  
- **1** = Fraudulent transaction  

We expect a fraud rate between 0.8% and 1.5%, consistent with real-world issuer statistics (Nilson Report, 2023).

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_ratio = fraud_counts[1] / len(df) * 100
imbalance_ratio = fraud_counts[0] / fraud_counts[1]

print("Class Distribution:")
print(f"  Legitimate (0): {fraud_counts[0]:>6,}  ({100 - fraud_ratio:.2f}%)")
print(f"  Fraudulent (1): {fraud_counts[1]:>6,}  ({fraud_ratio:.2f}%)")
print(f"\nImbalance Ratio: {imbalance_ratio:.1f}:1")
print(f"\nInterpretation: For every 1 fraudulent transaction, there are ~{imbalance_ratio:.0f} legitimate ones.")
print("This confirms severe class imbalance requiring SMOTE or class-weighted loss functions.")

### 3. Feature Descriptions

| Feature | Type | Description |
|:--------|:-----|:------------|
| `amount` | Continuous | Transaction amount in USD |
| `distance` | Continuous | Geographic distance from cardholder's home location (km) |
| `time_delta` | Continuous | Hours since the cardholder's previous transaction |
| `merchant_risk` | Continuous [0,1] | Risk score of the merchant category (luxury, gambling, electronics = high) |
| `device_trust` | Continuous [0,1] | Trust score of the originating device/browser |
| `velocity_1h` | Integer | Number of transactions in the last 1 hour |
| `velocity_24h` | Integer | Number of transactions in the last 24 hours |
| `hour_of_day` | Integer [0-23] | Hour when the transaction occurred |
| `is_weekend` | Binary | 1 if Saturday/Sunday, 0 otherwise |
| `is_fraud` | Binary | Target label (0 = legit, 1 = fraud) |

### 4. Summary Statistics by Class

Comparing the mean values of each feature between legitimate and fraudulent transactions reveals the statistical separation that ML models can exploit.

In [ ]:
numeric_cols = ['amount', 'distance', 'time_delta', 'merchant_risk', 'device_trust', 'velocity_1h', 'velocity_24h']

comparison = df.groupby('is_fraud')[numeric_cols].agg(['mean', 'std', 'median']).T
comparison.columns = ['Legit_Mean', 'Legit_Std', 'Legit_Median', 'Fraud_Mean', 'Fraud_Std', 'Fraud_Median']
comparison['Ratio (Fraud/Legit)'] = comparison['Fraud_Mean'] / comparison['Legit_Mean']
print("Feature Comparison: Legitimate vs Fraudulent Transactions")
print("=" * 80)
comparison.round(4)

**Key Observations:**
- **Distance:** Fraudulent transactions originate significantly farther from the cardholder's home (impossible travel).
- **Device Trust:** Fraud correlates with low device trust scores (untrusted browsers, VPNs, new devices).
- **Time Delta:** Fraudulent transactions occur in rapid succession (velocity attacks).
- **Amount:** Fraud transactions tend toward higher values, but this alone is insufficient for separation.
- **Merchant Risk:** Elevated merchant risk categories strongly correlate with fraud.

### 5. Correlation Analysis

In [ ]:
all_numeric = numeric_cols + ['hour_of_day', 'is_weekend', 'is_fraud']
corr = df[all_numeric].corr()

print("Pearson Correlation with is_fraud (sorted by absolute value):")
print("=" * 60)
fraud_corr = corr['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False)
for feat, val in fraud_corr.items():
    direction = 'POSITIVE' if val > 0 else 'NEGATIVE'
    bar = '#' * int(abs(val) * 40)
    print(f"  {feat:<18} {val:+.4f}  {direction:<8}  {bar}")

### 6. Distributional Analysis: Amount by Class

In [ ]:
print("Transaction Amount Statistics by Class:")
print("=" * 60)
for label, name in [(0, 'Legitimate'), (1, 'Fraudulent')]:
    subset = df[df['is_fraud'] == label]['amount']
    print(f"\n  {name} Transactions (N={len(subset):,}):")
    print(f"    Min:    ${subset.min():>10,.2f}")
    print(f"    Q1:     ${subset.quantile(0.25):>10,.2f}")
    print(f"    Median: ${subset.median():>10,.2f}")
    print(f"    Q3:     ${subset.quantile(0.75):>10,.2f}")
    print(f"    Max:    ${subset.max():>10,.2f}")
    print(f"    Mean:   ${subset.mean():>10,.2f}")
    print(f"    Std:    ${subset.std():>10,.2f}")

### 7. Temporal Patterns

In [ ]:
print("Fraud Rate by Hour of Day:")
print("=" * 50)
hourly = df.groupby('hour_of_day')['is_fraud'].agg(['sum', 'count'])
hourly['rate_pct'] = (hourly['sum'] / hourly['count'] * 100).round(2)
for hour, row in hourly.iterrows():
    bar = '#' * int(row['rate_pct'] * 5)
    print(f"  Hour {hour:>2}: {row['rate_pct']:>5.2f}%  ({int(row['sum']):>3} / {int(row['count']):>4})  {bar}")

In [ ]:
print("\nFraud Rate: Weekday vs Weekend:")
print("=" * 40)
weekend = df.groupby('is_weekend')['is_fraud'].agg(['sum', 'count'])
weekend['rate_pct'] = (weekend['sum'] / weekend['count'] * 100).round(2)
for label, row in weekend.iterrows():
    name = 'Weekend' if label == 1 else 'Weekday'
    print(f"  {name:<8}: {row['rate_pct']:.2f}% fraud  ({int(row['sum'])} / {int(row['count'])})")

### 8. Missing Values and Data Quality Check

In [ ]:
missing = df.isnull().sum()
print("Missing Values per Column:")
print(missing[missing > 0] if missing.sum() > 0 else "  None (dataset is complete)")
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
print(f"Unique Transaction Records: {len(df):,}")

### 9. EDA Conclusions and Modeling Strategy

**Findings:**
1. The dataset exhibits **~1.2% fraud rate** (imbalance ratio ~82:1), confirming the need for SMOTE oversampling on the training split.
2. **Distance from home** and **device trust score** show the strongest linear correlation with fraud, but non-linear interactions (e.g., high distance + low device trust + high amount) create composite fraud signatures.
3. Fraud is **temporally concentrated** during late-night / early-morning hours.
4. The dataset is clean with no missing values or duplicates.

**Modeling Strategy:**
- Use **Precision-Recall AUC** (not ROC-AUC alone) as the primary evaluation metric, since PR-AUC is more informative under severe class imbalance.
- Apply **RobustScaler** (robust to outliers) on the training split only.
- Apply **SMOTE** on the training split only to prevent data leakage.
- Compare linear (Logistic Regression), ensemble (Random Forest), and gradient boosting (XGBoost) algorithms.
- Calibrate operational thresholds for a 3-tier action system rather than using a fixed 0.50 cutoff.

---
*Proceed to Notebook 02: Data Preprocessing and Leakage Prevention.*